# Mini Semantic Search

**A beginner-friendly introduction to semantic search using Sentence Transformers and ChromaDB**

---

## What this notebook does

This notebook walks through every step of building a semantic search system from scratch:

1. Load a dataset of Frequently Asked Questions (FAQs) about programming.
2. Clean and validate the data.
3. Split long answers into smaller **chunks** so each piece of text is focused.
4. Convert each chunk into a **numerical vector** (embedding) using a pretrained AI model.
5. Store all vectors in **ChromaDB** — a lightweight vector database.
6. Accept a natural-language question and retrieve the **top 3 most relevant answers**.

---

## Why semantic search?

Traditional keyword search only finds documents that contain the exact words you typed.

Semantic search understands **meaning**. If you ask _"How do I save my code online?"_ it can still match _"Upload your project to GitHub"_ — because both phrases express the same idea, even though they share no keywords.

Semantic search works by representing text as points in a high-dimensional space where similar meanings are close together. Finding relevant answers is then a matter of finding the nearest points to your query.

---

## Key concepts

| Concept | Plain English explanation |
|---|---|
| **Embedding** | A list of numbers (a vector) that represents the meaning of a piece of text. |
| **Sentence Transformer** | A pretrained AI model that converts text into embeddings. |
| **Vector database** | A database optimised for storing and searching vectors by similarity. |
| **Chunking** | Splitting a long text into smaller overlapping pieces for more precise retrieval. |
| **Cosine distance** | A measure of how different two vectors are. 0 = identical, higher = less similar. |

---

> **Note:** The first time you run this notebook it will download the embedding model (~90 MB) from HuggingFace. An internet connection is required for that one-time download. Subsequent runs use the local cache.

## 1. Imports and configuration

We import the project modules from the `src/` directory and set up the paths we need.

In [ ]:
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Make sure the project root is on sys.path so we can import from src/
# This works whether you run the notebook from the project root or from
# the notebooks/ subdirectory.
# ---------------------------------------------------------------------------
notebook_dir = Path().resolve()                    # current working directory
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python path includes project root: {str(project_root) in sys.path}")

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_FILE  = project_root / 'data' / 'faqs.csv'
CHROMA_DIR = project_root / 'chroma_db'

print(f"\nData file : {DATA_FILE}")
print(f"Data exists: {DATA_FILE.exists()}")
print(f"ChromaDB dir: {CHROMA_DIR}")

In [ ]:
# Import our project modules
from src.data_loader   import load_csv
from src.text_cleaner  import clean_dataframe
from src.chunker       import chunk_dataframe, chunk_faq_row
from src.semantic_search import SemanticSearch

print("All project modules imported successfully.")

## 2. Load the FAQ dataset

We load the CSV file using our `load_csv` function, which validates the data and returns a clean pandas DataFrame.

In [ ]:
# Load the sample FAQ dataset
df_raw = load_csv(DATA_FILE)

print(f"\nLoaded {len(df_raw)} FAQ rows.")
print(f"Columns: {list(df_raw.columns)}")

## 3. Inspect the dataset

Before processing, it is always good practice to look at the data.

In [ ]:
# Show the first few rows
df_raw.head(5)

In [ ]:
# Dataset shape: (rows, columns)
print(f"Shape: {df_raw.shape}")

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df_raw.isnull().sum())

In [ ]:
# Look at the data types
df_raw.info()

In [ ]:
# Show the length of each answer (in characters) to understand variability
df_raw['answer_length'] = df_raw['answer'].str.len()
print("Answer length statistics (characters):")
print(df_raw['answer_length'].describe())

# The longest answer
longest_idx = df_raw['answer_length'].idxmax()
print(f"\nLongest answer belongs to FAQ id={df_raw.loc[longest_idx, 'id']}:")
print(f"  Question: {df_raw.loc[longest_idx, 'question']}")
print(f"  Length:   {df_raw.loc[longest_idx, 'answer_length']} characters")

## 4. Clean and validate the dataset

The `clean_dataframe` function:
- Strips extra whitespace from questions and answers.
- Normalises line endings.
- Drops rows with an empty answer (they cannot be indexed usefully).
- Preserves technical terms, punctuation, and URLs.

In [ ]:
# Drop the helper column we added before cleaning
df_raw = df_raw.drop(columns=['answer_length'], errors='ignore')

# Clean the data
df = clean_dataframe(df_raw)

print(f"\nRows after cleaning: {len(df)}")
df.head(3)

## 5. Text chunking

Long answers are split into overlapping chunks so that each chunk is focused on one idea. This also helps ChromaDB retrieve a precise, relevant passage instead of a 500-word essay.

### Why character-based chunking?

We use a sliding window of **500 characters** with **50 characters of overlap**. This is simple and predictable. The overlap ensures that a sentence split across a boundary still appears in at least one complete chunk.

Short answers (under 500 characters) are kept as a single chunk.

In [ ]:
# Demonstrate chunking on one example FAQ
example_row = df[df['question'].str.contains('Git', case=False)].iloc[0]

print(f"FAQ question: {example_row['question']}")
print(f"Answer length: {len(example_row['answer'])} characters")
print(f"\nFull answer:\n{example_row['answer'][:300]}...")

print("\n" + "=" * 60)
print("Chunking this answer with chunk_size=200, overlap=30:")
print("=" * 60)

demo_chunks = chunk_faq_row(
    faq_id=int(example_row['id']),
    question=example_row['question'],
    answer=example_row['answer'],
    chunk_size=200,
    overlap=30,
)
for c in demo_chunks:
    print(f"\nChunk {c['chunk_index']} (id={c['chunk_id']}):")
    print(f"  [{len(c['text'])} chars] {c['text'][:120]}...")

In [ ]:
# Now chunk ALL FAQs with the default settings (500 chars, 50 overlap)
CHUNK_SIZE = 500
OVERLAP    = 50

chunks = chunk_dataframe(df, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

print(f"\nTotal chunks created: {len(chunks)}")
print(f"\nFirst chunk preview:")
import pprint
pprint.pprint(chunks[0])

## 6. Load the embedding model

We use `sentence-transformers/all-MiniLM-L6-v2` — a small, fast model that produces 384-dimensional embeddings. It is well-suited for a student laptop.

> **First-time download:** The model is approximately 90 MB and will be downloaded from HuggingFace automatically. Subsequent runs use the local cache.

In [ ]:
# Create a SemanticSearch instance — this sets up the model and ChromaDB
search_engine = SemanticSearch(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    collection_name='faq_chunks',
    db_dir=CHROMA_DIR,
)

# Trigger the model load now (lazy-loaded on first use, but let's be explicit)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Quick sanity check: embed a test sentence
test_embedding = model.encode(["Hello world"])
print(f"Model loaded. Embedding dimensions: {test_embedding.shape[1]}")
print("Expected: 384 dimensions for all-MiniLM-L6-v2")

## 7. Build the ChromaDB index

We embed every chunk and store it in ChromaDB. ChromaDB persists the data to the `chroma_db/` directory, so it survives restarts.

Setting `reset=True` clears any previous index first — useful when re-running the notebook from scratch.

In [ ]:
# Build the index (reset=True ensures a clean start when re-running)
search_engine.build_index(chunks, reset=True)

print(f"\nCollection size: {search_engine.collection_size()} chunks")

## 8. Run semantic search queries

Now the fun part. We ask natural-language questions and the system retrieves the most relevant FAQ answers.

### How results are scored

ChromaDB returns a **cosine distance** value:
- Distance **0.0** means the query and the chunk are identical in meaning.
- Distance **closer to 0** means more similar.
- Distance **closer to 2** means very different.

A distance below **0.5** is typically a strong match. We label scores as "distance (lower = more similar)" — not as a percentage or similarity score — to avoid confusion.

---

In [ ]:
# Query 1: GitHub upload
results = search_engine.search("How do I upload my code to GitHub?")

In [ ]:
# Query 2: List vs tuple
results = search_engine.search("What is the difference between a list and a tuple in Python?")

In [ ]:
# Query 3: Communicating with an API
results = search_engine.search("How can my application communicate with an API?")

In [ ]:
# Query 4: Storing data in a relational database
results = search_engine.search("How do I store information in a relational database?")

In [ ]:
# Query 5: Semantic / similarity search
results = search_engine.search("How can I find similar documents based on meaning?")

## 9. Demonstrate how changing the query changes results

One of the most powerful features of semantic search is that it handles paraphrases. Let's see the same question asked three different ways.

In [ ]:
print("=" * 70)
print("PARAPHRASE DEMONSTRATION")
print("Three different phrasings for the same underlying question.")
print("=" * 70)

queries = [
    "What is version control?",                      # direct terminology
    "How do I track changes in my project?",         # functional phrasing
    "Can I go back to a previous version of my code?",  # intent-based phrasing
]

for q in queries:
    results = search_engine.search(q, top_k=1)

In [ ]:
# Another example: virtual environments
print("=" * 70)
print("PARAPHRASE DEMONSTRATION — virtual environments")
print("=" * 70)

queries = [
    "What is a virtual environment?",
    "How do I isolate Python packages per project?",
    "My projects are using different versions of the same library",
]

for q in queries:
    results = search_engine.search(q, top_k=1)

## 10. Additional interesting queries

In [ ]:
# Machine learning
results = search_engine.search("What is the difference between supervised and unsupervised learning?")

In [ ]:
# Python decorators
results = search_engine.search("How do Python decorators work?")

In [ ]:
# REST API
results = search_engine.search("What HTTP methods should I use when designing a REST API?")

## 11. Conclusion and limitations

### What we built

We built a complete semantic search pipeline:

1. **Data loading** — validated CSV/JSON loading with Pandas.
2. **Text cleaning** — whitespace normalisation that preserves technical content.
3. **Chunking** — character-based sliding-window splitting with overlap.
4. **Embedding** — `all-MiniLM-L6-v2` converts text to 384-dimensional vectors.
5. **Indexing** — ChromaDB stores embeddings persistently with cosine distance.
6. **Retrieval** — natural-language queries return the top-k most relevant chunks.

---

### Limitations

| Limitation | Explanation |
|---|---|
| **Small dataset** | Only 24 FAQs. More data improves recall. |
| **Character chunking** | Words can be split at chunk boundaries. Word- or sentence-based chunking would be more linguistically clean. |
| **No re-ranking** | Results are ordered purely by embedding distance. A production system might add a cross-encoder re-ranker for precision. |
| **No UI** | This is a command-line/notebook demo. A real product would have a web frontend. |
| **Model limitations** | `all-MiniLM-L6-v2` is optimised for English. It may perform poorly on other languages. |

---

### Possible future improvements

- Sentence-based chunking using `nltk.sent_tokenize`.
- Support for larger document collections (PDF, web scraping).
- A simple web interface using Flask or Streamlit.
- Cross-encoder re-ranking for higher precision.
- Support for multilingual queries with a multilingual model.

---

*Built with Python 3.13, Sentence Transformers, ChromaDB, and Pandas.*